# T2.2 — Semantic Mapping Upload to DBRepo

Uploads all semantic mappings from `docs/semantic_mapping.csv` to DBRepo via REST API.

**Owner:** Person B | **Task:** T2.2 — Semantic Mapping | **Dataset:** Hohe Warte Vienna Weather

> **Requires:** TU Wien VPN active, and the group database already created in DBRepo (T2.1 complete).

## Step 0 — Install the official DBRepo Python library

In [4]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'dbrepo', '--quiet'])
print('dbrepo library ready')


dbrepo library ready


## Step 1 — Configuration

In [ ]:
ENDPOINT    = "https://dbrepo1.ec.tuwien.ac.at"            # TU Wien DBRepo instance
DATABASE_ID = "899bfcba-7fec-40c9-9076-3a3a9372c844"       # your group database UUID
USERNAME    = "azra1558"                                    # DBRepo username
PASSWORD    = "katalizator1558!"                          # fill in locally — never commit

MAPPING_FILE = "../docs/semantic_mapping.csv"


## Step 2 — Connect and inspect the database structure

We need the **numeric table ID** and **numeric column ID** for every column — the API does not accept names.

In [6]:
from dbrepo.RestClient import RestClient

client = RestClient(endpoint=ENDPOINT, username=USERNAME, password=PASSWORD)

db = client.get_database(database_id=DATABASE_ID)
print(f"Connected to database: {db.name}")
print(f"Tables found: {[t.name for t in db.tables]}")


ConnectTimeout: HTTPSConnectionPool(host='dbrepo1.ec.tuwien.ac.at', port=443): Max retries exceeded with url: /api/v1/database/899bfcba-7fec-40c9-9076-3a3a9372c844 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x0000025027213110>, 'Connection to dbrepo1.ec.tuwien.ac.at timed out. (connect timeout=None)'))

## Step 3 — Build name-to-ID lookup maps

In [ ]:
table_id_map = {}  
column_id_map = {} 

for table in db.tables:
    table_id_map[table.name] = table.id
    print(f"  Table '{table.name}' -> id={table.id}")
    for col in table.columns:
        column_id_map[(table.name, col.name)] = col.id
        print(f"    Column '{col.name}' -> id={col.id}")

print(f"\nMapped {len(table_id_map)} tables, {len(column_id_map)} columns")


## Step 4 — Load the semantic mapping CSV

In [ ]:
import csv

mappings = []
with open(MAPPING_FILE, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        mappings.append(row)

print(f"Loaded {len(mappings)} mappings from CSV")
for m in mappings[:3]:
    print(m)


## Step 5 — Upload each semantic concept via REST API

Endpoint: `PUT /api/v1/database/{databaseId}/table/{tableId}/column/{columnId}`

In [ ]:
import requests

auth = requests.post(
    f"{ENDPOINT}/api/v1/user/login",
    json={"username": USERNAME, "password": PASSWORD}
)
if auth.status_code != 200:
    # Fallback: try alternate auth endpoint
    auth = requests.post(
        f"{ENDPOINT}/api/v1/authenticate",
        json={"username": USERNAME, "password": PASSWORD}
    )
if auth.status_code != 200:
    raise RuntimeError(f"Login failed {auth.status_code}: {auth.text}")

token_data = auth.json()
TOKEN = token_data.get("token") or token_data.get("access_token") or token_data.get("jwt")
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
print("Authenticated successfully")

# Upload
success, skipped, errors = 0, 0, []

for row in mappings:
    tname = row["table_name"]
    cname = row["column_name"]
    uri   = row["ontology_uri"]
    label = row["ontology_label"]

    # Resolve numeric IDs
    tid = table_id_map.get(tname)
    cid = column_id_map.get((tname, cname))

    if tid is None or cid is None:
        skipped += 1
        print(f"  SKIP {tname}.{cname} — not found in DBRepo (check table/column name)")
        continue

    url = f"{ENDPOINT}/api/v1/database/{DATABASE_ID}/table/{tid}/column/{cid}"
    payload = {"concept_uri": uri, "concept_name": label}

    resp = requests.put(url, json=payload, headers=HEADERS)

    if resp.status_code in (200, 201, 204):
        success += 1
        print(f"  OK   {tname}.{cname}")
    else:
        errors.append((tname, cname, resp.status_code, resp.text))
        print(f"  FAIL {tname}.{cname} -> HTTP {resp.status_code}: {resp.text[:120]}")

print()
print(f"Result: {success} uploaded, {skipped} skipped, {len(errors)} failed")
if errors:
    for e in errors:
        print(f"  {e[0]}.{e[1]}: HTTP {e[2]}")


## Step 6 — Verify: read back a sample of columns from DBRepo

In [ ]:
spot_checks = [
    ("weather_measurement", "t_mean_c"),
    ("weather_measurement", "precp_sum_mm"),
    ("station",             "nuts_code"),
    ("station",             "latitude_deg"),
    ("time_dimension",      "ref_year"),
]

print("Spot-check verification:")
all_ok = True
for tname, cname in spot_checks:
    tid = table_id_map.get(tname)
    cid = column_id_map.get((tname, cname))
    if tid is None or cid is None:
        print(f"  SKIP {tname}.{cname} — ID not found")
        continue
    url = f"{ENDPOINT}/api/v1/database/{DATABASE_ID}/table/{tid}/column/{cid}"
    resp = requests.get(url, headers=HEADERS)
    if resp.status_code == 200:
        data = resp.json()
        uri = data.get("concept_uri") or data.get("uri") or "(field name differs)"
        print(f"  OK   {tname}.{cname}")
        print(f"       -> {uri}")
    else:
        print(f"  FAIL {tname}.{cname} -> HTTP {resp.status_code}")
        all_ok = False

print()
print("All spot-checks passed" if all_ok else "Some checks failed — review above")
